# Build the guide-module object

**Run this notebook before `Figure3_B`.** It writes
`outputs/adata_SingleKO_GuideModules.h5ad`, which `Figure3_B` reads.

`Figure3_B` needs one thing the screen object does not carry: a per-cell
indicator of which *guide module* a cell was perturbed in (`K_0` … `K_5`, and
`K_CONTROL` for control cells). Those columns were originally added by a
separate pipeline, which is why that figure used to read an object no other
notebook touched.

This notebook derives them from
`adata-hash-features_singlets_05242020.h5ad` — the same screen object the
other figure notebooks read — following the recipe in the analysis pipeline
(`Notebooks/07_06_ReduceToGenes.ipynb` and
`Notebooks/08-01-ReduceAnndataToSelectedKOsGenes.ipynb`):

1. restrict genes to the 1,041 used for the module analysis
2. drop guides seen in fewer than 20 cells, and guides on the bad-guide list
3. aggregate the surviving guides to their target gene
4. keep cells carrying guides for exactly one module gene, or a control
5. record which module that gene belongs to

The UMAP coordinates and the leiden labels are carried through unchanged, so
downstream figures sit in the same embedding as before.

## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc

sc.settings.verbosity = "hint"

In [2]:
E3LIGASE = Path("/home/eraslab1/Projects/E3Ligase/analysisSingle")

SCREEN_H5AD    = E3LIGASE / "outputs/anndata/OriginalFiles/adata-hash-features_singlets_05242020.h5ad"
GENE_MODULES_8 = E3LIGASE / "TextFiles/ME_GeneModules_leiden_8_Modules.csv"
GUIDE_MODULES  = E3LIGASE / "TextFiles/ME_GuideModules_leiden_6_Modules.csv"
BAD_KO_GUIDES  = E3LIGASE / "R/GuideSelect_BadKOGuides.csv"

OUTPUT_DIR = Path("outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_H5AD = OUTPUT_DIR / "adata_SingleKO_GuideModules.h5ad"

CONTROL_PREFIXES = ("NO_TARGET_", "ONE_NONGENE_SITE_")
MIN_CELLS_PER_GUIDE = 20      # guides seen in fewer cells are dropped

# genes added to the module list by the original pipeline
EXTRA_GENES = [
    "0610012G03Rik", "2010005H15Rik", "2010111I01Rik", "2310001H17Rik",
    "2810474O19Rik", "H2-Q7", "H2-Q6", "H2-DMa", "H2-T23", "H2-DMb1",
    "H2-Ab1", "H2-Aa", "H2-Eb1", "H2-M2", "H2-K1", "H2-D1",
]

## Load the screen and restrict to the module gene set

In [3]:
adata = sc.read(SCREEN_H5AD)
print(f"screen: {adata.shape[0]} cells x {adata.shape[1]} genes")

moduleGenes = list(pd.read_csv(GENE_MODULES_8, index_col=0).iloc[:, 0])
keepGenes = [g for g in moduleGenes + EXTRA_GENES if g in adata.var_names]
adata = adata[:, keepGenes].copy()
print(f"reduced to {adata.shape[1]} genes")

/home/eraslab1/miniconda3/lib/python3.8/site-packages/anndata/compat/__init__.py:229: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(
/home/eraslab1/miniconda3/lib/python3.8/site-packages/anndata/compat/__init__.py:229: FutureWarning: Moving element from .uns['neighbors']['connectivities'] to .obsp['connectivities'].

This is where adjacency matrices should go now.
  warn(


screen: 519535 cells x 13811 genes
reduced to 1041 genes


## Drop unreliable guides

A guide seen in only a handful of cells cannot support an effect estimate, and
the pipeline additionally curated a list of knockout guides that did not work.

In [4]:
guideColumns = [c for c in adata.obs.columns if c in set(adata.uns["feature_barcode_names"])]
guides = adata.obs[guideColumns] > 0            # binarise: did the cell receive this guide

badGuides = set(pd.read_csv(BAD_KO_GUIDES)["x"])
tooFew = guides.sum(axis=0) < MIN_CELLS_PER_GUIDE
drop = tooFew | guides.columns.isin(badGuides)

print(f"guides total       : {guides.shape[1]}")
print(f"  seen in <{MIN_CELLS_PER_GUIDE} cells : {int(tooFew.sum())}")
print(f"  on the bad list  : {int(guides.columns.isin(badGuides).sum())}")
guides = guides.loc[:, ~drop]
print(f"  kept             : {guides.shape[1]}")

guides total       : 3720
  seen in <20 cells : 64
  on the bad list  : 941
  kept             : 2715


## Aggregate guides to their target gene

A cell counts as perturbed in a gene if it received any surviving guide against
it. A cell counts as a control if its dominant guide is a control guide.

In [5]:
isControl = np.array([c.startswith(CONTROL_PREFIXES) for c in guides.columns])
hasGuide = guides.any(axis=1)

targetGene = pd.Series(
    [c.rsplit("_", 1)[0] for c in guides.columns], index=guides.columns
)
perGene = guides.loc[:, ~isControl].groupby(targetGene[~isControl], axis=1).any()

# control assignment follows the pipeline: the cell's dominant guide decides
dominantIsControl = pd.Series(isControl[guides.values.argmax(axis=1)], index=guides.index)
perGene["CONTROL"] = dominantIsControl & hasGuide

print(f"cells with at least one surviving guide: {int(hasGuide.sum())}")
print(f"target genes after aggregation         : {perGene.shape[1] - 1}")

cells with at least one surviving guide: 428213
target genes after aggregation         : 1069


## Keep cells perturbed in exactly one module gene

Cells carrying a guide against a gene outside the module set are dropped, as
are cells carrying more than one target, so each remaining cell belongs to a
single module.

In [6]:
guideModules = pd.read_csv(GUIDE_MODULES, index_col=0)
moduleOf = dict(zip(guideModules.GuideName, guideModules.GuideGroup))

inModule = [g for g in perGene.columns if g in moduleOf]
outsideModule = [g for g in perGene.columns if g not in moduleOf and g != "CONTROL"]

nTargets = perGene[inModule + ["CONTROL"]].sum(axis=1)
keep = hasGuide & (nTargets > 0) & (perGene[outsideModule].sum(axis=1) == 0)
singleTarget = keep & (nTargets == 1)

print(f"module genes                 : {len(inModule)}")
print(f"cells with only module guides: {int(keep.sum())}")
print(f"  of which a single target   : {int(singleTarget.sum())}")

module genes                 : 329
cells with only module guides: 181787
  of which a single target   : 170393


## Record the module of each cell and save

In [7]:
for module in sorted(set(moduleOf.values())):
    genesInModule = [g for g in inModule if moduleOf[g] == module]
    adata.obs[f"K_{module}"] = (perGene[genesInModule].any(axis=1) & singleTarget).astype(int)
adata.obs["K_CONTROL"] = (perGene["CONTROL"] & singleTarget).astype(int)

adata = adata[singleTarget.values].copy()

moduleColumns = [f"K_{m}" for m in sorted(set(moduleOf.values()))] + ["K_CONTROL"]
print(adata.obs[moduleColumns].sum().to_string())
print(f"\nfinal object: {adata.shape[0]} cells x {adata.shape[1]} genes")
print(f"UMAP carried through: {'X_umap' in adata.obsm}")

K_0          31903
K_1          25219
K_2          23512
K_3          24037
K_4           6685
K_5           2805
K_CONTROL    56232

final object: 170393 cells x 1041 genes
UMAP carried through: True


In [8]:
adata.write(OUTPUT_H5AD)
print(f"written to {OUTPUT_H5AD.resolve()}")

written to /home/eraslab1/Projects/PerturbDecode/notebooks/manuscript_figures/outputs/adata_SingleKO_GuideModules.h5ad
